In [1]:
# This notebook is 
#get common train/validation/test data
import pandas as pd
train_df = pd.read_csv("../covid/splits/train.csv")
val_df = pd.read_csv("../covid/splits/validation.csv")
test_df = pd.read_csv("../covid/splits/test.csv")

In [2]:
train_gen_df = train_df.copy()
val_gen_df = val_df.copy()
test_gen_df = test_df.copy()

train_gen_df["target"] = train_gen_df["target"].astype(str)
val_gen_df["target"] = val_gen_df["target"].astype(str)
test_gen_df["target"] = test_gen_df["target"].astype(str)

In [3]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input

# TRAIN : preprocessing VGG16 + Data Augmentation
train_datagen_e6 = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)

# VALIDATION : preprocessing VGG16 uniquement
val_datagen_e6 = ImageDataGenerator(
    preprocessing_function=preprocess_input
)


BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
COLOR_MODE = "rgb"

train_generator_e6 = train_datagen_e6.flow_from_dataframe(
    dataframe=train_gen_df,
    x_col="filepath",
    y_col="target",
    target_size=IMAGE_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary",
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

val_generator_e6 = val_datagen_e6.flow_from_dataframe(
    dataframe=val_gen_df,
    x_col="filepath",
    y_col="target",
    target_size=IMAGE_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary",
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_generator_e6 = val_datagen_e6.flow_from_dataframe(
    dataframe=test_gen_df,
    x_col="filepath",
    y_col="target",
    target_size=IMAGE_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary",
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 13545 validated image filenames belonging to 2 classes.
Found 3387 validated image filenames belonging to 2 classes.
Found 4233 validated image filenames belonging to 2 classes.


In [4]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout

base_model_e6 = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model_e6.trainable = False

model_e6 = Sequential([
    base_model_e6,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

In [5]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model_e6.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

early_stopping_e6 = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

checkpoint_e6 = ModelCheckpoint(
    "E6_VGG16_TL_DA_best.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

In [ ]:
history_e6 = model_e6.fit(
    train_generator_e6,
    validation_data=val_generator_e6,
    epochs=5,
    callbacks=[early_stopping_e6, checkpoint_e6],
    verbose=1
)

Epoch 1/5
  4/424 ━━━━━━━━━━━━━━━━━━━━ 43:03 6s/step - accuracy: 0.7891 - auc: 0.7491 - loss: 0.6821 - precision: 0.4500 - recall: 0.3600   